# Create Provider Ground Truth

This notebook creates the provider ground-truth table for the MDM project.

The source is the CMS NPPES provider file. From that file, the notebook keeps active individual providers with a Texas practice location, pulls the provider's primary taxonomy and license information, and selects a reproducible sample of 3,000 providers.

The final CSV will act as the known source of truth when the EHR, HR, and credentialing datasets are simulated. Each simulated record can later be traced back to a `ground_truth_id` so matching results can be evaluated against the provider's real identity.


## 1. Setup

Import the libraries used in the notebook and define the source, output, and sampling settings in one place.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd


NPPES_DIR = Path(
    "../data/raw/nppes/NPPES_Data_Dissemination_August_2026_V2"
)

GROUND_TRUTH_DIR = Path("../data/ground_truth")
GROUND_TRUTH_DIR.mkdir(parents=True, exist_ok=True)

GROUND_TRUTH_SIZE = 3000
CHUNK_SIZE = 100_100
RANDOM_SEED = 42

rng = np.random.default_rng(RANDOM_SEED)


## 2. Locate the main NPPES provider file

The NPPES download folder contains multiple files. For the ground-truth table, this notebook uses the main `npidata_pfile` CSV.


In [2]:
nppes_files = list(NPPES_DIR.glob("npidata_pfile_*.csv"))

if len(nppes_files) != 1:
    raise ValueError(
        f"Expected 1 main NPPES file, found {len(nppes_files)}: {nppes_files}"
    )

NPPES_FILE = nppes_files[0]

print(f"Using NPPES file: {NPPES_FILE.name}")


Using NPPES file: npidata_pfile_20050523-20260809.csv


## 3. Inspect the source schema

The full NPPES file is very large, so only the header is loaded here. This lets us review the available fields without reading the full dataset into memory.


In [3]:
nppes_header = pd.read_csv(
    NPPES_FILE,
    nrows=0,
    dtype=str
)

print(f"Number of columns: {len(nppes_header.columns)}")


Number of columns: 330


In [4]:
# Optional schema review
for column in nppes_header.columns:
    print(column)


NPI
Entity Type Code
Replacement NPI
Employer Identification Number (EIN)
Provider Organization Name (Legal Business Name)
Provider Last Name (Legal Name)
Provider First Name
Provider Middle Name
Provider Name Prefix Text
Provider Name Suffix Text
Provider Credential Text
Provider Other Organization Name
Provider Other Organization Name Type Code
Provider Other Last Name
Provider Other First Name
Provider Other Middle Name
Provider Other Name Prefix Text
Provider Other Name Suffix Text
Provider Other Credential Text
Provider Other Last Name Type Code
Provider First Line Business Mailing Address
Provider Second Line Business Mailing Address
Provider Business Mailing Address City Name
Provider Business Mailing Address State Name
Provider Business Mailing Address Postal Code
Provider Business Mailing Address Country Code (If outside U.S.)
Provider Business Mailing Address Telephone Number
Provider Business Mailing Address Fax Number
Provider First Line Business Practice Location Address
P

## 4. Define the fields needed for ground truth

The ground-truth table only needs a subset of the NPPES fields:

- provider identity
- practice location and phone
- NPI status dates
- taxonomy and license information

NPPES stores up to 15 taxonomy/license groups for a provider, so those repeated fields are generated programmatically.


In [5]:
base_columns = [
    "NPI",
    "Entity Type Code",

    "Provider Last Name (Legal Name)",
    "Provider First Name",
    "Provider Middle Name",
    "Provider Name Prefix Text",
    "Provider Name Suffix Text",
    "Provider Credential Text",

    "Provider First Line Business Practice Location Address",
    "Provider Second Line Business Practice Location Address",
    "Provider Business Practice Location Address City Name",
    "Provider Business Practice Location Address State Name",
    "Provider Business Practice Location Address Postal Code",
    "Provider Business Practice Location Address Telephone Number",

    "Provider Enumeration Date",
    "Last Update Date",
    "NPI Deactivation Date",
    "NPI Reactivation Date",
]

taxonomy_columns = []

for i in range(1, 16):
    taxonomy_columns.extend([
        f"Healthcare Provider Taxonomy Code_{i}",
        f"Provider License Number_{i}",
        f"Provider License Number State Code_{i}",
        f"Healthcare Provider Primary Taxonomy Switch_{i}",
    ])

columns_to_load = base_columns + taxonomy_columns


In [6]:
missing_columns = [
    column
    for column in columns_to_load
    if column not in nppes_header.columns
]

if missing_columns:
    raise ValueError(f"Missing required NPPES columns: {missing_columns}")

print(f"All {len(columns_to_load)} required columns are available.")


All 78 required columns are available.


## 5. Extract each provider's primary taxonomy

A provider can have several taxonomy codes in NPPES. The field `Healthcare Provider Primary Taxonomy Switch_#` identifies which taxonomy is primary.

For the primary taxonomy entry, the helper below also keeps the related license number and license state.


In [7]:
def extract_primary_taxonomy(df):
    primary_taxonomy = pd.Series(pd.NA, index=df.index, dtype="string")
    primary_license = pd.Series(pd.NA, index=df.index, dtype="string")
    primary_license_state = pd.Series(pd.NA, index=df.index, dtype="string")

    for i in range(1, 16):
        taxonomy_col = f"Healthcare Provider Taxonomy Code_{i}"
        license_col = f"Provider License Number_{i}"
        license_state_col = f"Provider License Number State Code_{i}"
        primary_switch_col = f"Healthcare Provider Primary Taxonomy Switch_{i}"

        is_primary = (
            df[primary_switch_col]
            .fillna("")
            .str.strip()
            .str.upper()
            .eq("Y")
        )

        primary_taxonomy.loc[is_primary] = df.loc[is_primary, taxonomy_col]
        primary_license.loc[is_primary] = df.loc[is_primary, license_col]
        primary_license_state.loc[is_primary] = df.loc[
            is_primary, license_state_col
        ]

    return primary_taxonomy, primary_license, primary_license_state


## 6. Build the candidate provider pool

The NPPES file is read in chunks so the full file does not need to be loaded into memory at once.

A provider is eligible for the ground-truth sample when they:

1. are an individual provider (`Entity Type Code = 1`)
2. have a Texas practice location
3. are treated as active based on the NPI deactivation/reactivation fields
4. have an NPI, first name, and last name

After filtering each chunk, a reproducible 10% sample is kept. This keeps the candidate pool manageable before selecting the final 3,000 providers.


In [ ]:
candidate_chunks = []

for chunk in pd.read_csv(
    NPPES_FILE,
    usecols=columns_to_load,
    dtype=str,
    chunksize=CHUNK_SIZE,
    low_memory=False
):
    # Individual providers only
    individual_mask = (
        chunk["Entity Type Code"]
        .fillna("")
        .str.strip()
        .eq("1")
    )

    # Texas practice location
    texas_mask = (
        chunk["Provider Business Practice Location Address State Name"]
        .fillna("")
        .str.strip()
        .str.upper()
        .eq("TX")
    )

    # Treat providers as active if they were never deactivated,
    # or if they were later reactivated.
    deactivation_date = pd.to_datetime(
        chunk["NPI Deactivation Date"],
        format="%m/%d/%Y",
        errors="coerce"
    )

    reactivation_date = pd.to_datetime(
        chunk["NPI Reactivation Date"],
        format="%m/%d/%Y",
        errors="coerce"
    )

    active_mask = (
        deactivation_date.isna()
        | (
            reactivation_date.notna()
            & (reactivation_date >= deactivation_date)
        )
    )

    # Require the basic fields needed to identify a provider
    identity_mask = (
        chunk["NPI"].notna()
        & chunk["Provider First Name"].notna()
        & chunk["Provider Last Name (Legal Name)"].notna()
    )

    filtered = chunk.loc[
        individual_mask
        & texas_mask
        & active_mask
        & identity_mask
    ].copy()

    if filtered.empty:
        continue

    (
        filtered["primary_taxonomy_code"],
        filtered["primary_license_number"],
        filtered["primary_license_state"]
    ) = extract_primary_taxonomy(filtered)

    # Keep a smaller random subset from each chunk
    random_values = rng.random(len(filtered))
    filtered = filtered.loc[random_values < 0.10].copy()

    if not filtered.empty:
        candidate_chunks.append(filtered)


In [ ]:
candidates = pd.concat(
    candidate_chunks,
    ignore_index=True
)

print(f"Candidate providers: {len(candidates):,}")


In [ ]:
candidates.head()


## 7. Select the final ground-truth sample

Select exactly 3,000 providers from the candidate pool. A fixed random seed keeps the sample reproducible between runs.


In [ ]:
ground_truth = (
    candidates
    .sample(
        n=GROUND_TRUTH_SIZE,
        random_state=RANDOM_SEED
    )
    .reset_index(drop=True)
)

print(f"Selected ground-truth providers: {len(ground_truth):,}")


## 8. Format the ground-truth table

Keep only the fields that will be useful for the MDM simulation and rename the NPPES column names to shorter project-friendly names.


In [ ]:
ground_truth = ground_truth[[
    "NPI",
    "Entity Type Code",

    "Provider First Name",
    "Provider Middle Name",
    "Provider Last Name (Legal Name)",
    "Provider Name Prefix Text",
    "Provider Name Suffix Text",
    "Provider Credential Text",

    "Provider First Line Business Practice Location Address",
    "Provider Second Line Business Practice Location Address",
    "Provider Business Practice Location Address City Name",
    "Provider Business Practice Location Address State Name",
    "Provider Business Practice Location Address Postal Code",
    "Provider Business Practice Location Address Telephone Number",

    "primary_taxonomy_code",
    "primary_license_number",
    "primary_license_state",

    "Provider Enumeration Date",
    "Last Update Date"
]].copy()


In [ ]:
ground_truth = ground_truth.rename(columns={
    "NPI": "npi",
    "Entity Type Code": "entity_type_code",

    "Provider First Name": "first_name",
    "Provider Middle Name": "middle_name",
    "Provider Last Name (Legal Name)": "last_name",
    "Provider Name Prefix Text": "name_prefix",
    "Provider Name Suffix Text": "name_suffix",
    "Provider Credential Text": "credential",

    "Provider First Line Business Practice Location Address":
        "practice_address_line_1",

    "Provider Second Line Business Practice Location Address":
        "practice_address_line_2",

    "Provider Business Practice Location Address City Name":
        "practice_city",

    "Provider Business Practice Location Address State Name":
        "practice_state",

    "Provider Business Practice Location Address Postal Code":
        "practice_zip",

    "Provider Business Practice Location Address Telephone Number":
        "practice_phone",

    "Provider Enumeration Date": "enumeration_date",
    "Last Update Date": "last_update_date"
})


### Add project-level identifiers

`ground_truth_id` is the internal ID that will tie the future simulated source records back to the same real provider.

Because inactive providers were removed during filtering, the selected records are labeled `ACTIVE`.


In [ ]:
ground_truth.insert(
    0,
    "ground_truth_id",
    [
        f"GT{i:05d}"
        for i in range(1, len(ground_truth) + 1)
    ]
)

ground_truth["npi_status"] = "ACTIVE"


## 9. Validate the ground truth

Before saving the file, check the conditions that should always be true:

- exactly 3,000 providers
- no duplicate NPIs
- no duplicate ground-truth IDs
- all records are individual providers
- all practice locations are in Texas

Missing values are reviewed separately because fields such as middle name, suffix, second address line, credential, and license information are not expected to be complete for every provider.


In [ ]:
assert len(ground_truth) == GROUND_TRUTH_SIZE
assert ground_truth["npi"].duplicated().sum() == 0
assert ground_truth["ground_truth_id"].duplicated().sum() == 0
assert ground_truth["entity_type_code"].eq("1").all()
assert ground_truth["practice_state"].eq("TX").all()

print("Core validation checks passed.")


In [ ]:
validation_summary = pd.Series({
    "rows": len(ground_truth),
    "unique_npis": ground_truth["npi"].nunique(),
    "duplicate_npis": ground_truth["npi"].duplicated().sum(),
    "unique_ground_truth_ids": ground_truth["ground_truth_id"].nunique(),
})

validation_summary


In [ ]:
ground_truth.isna().sum().sort_values(ascending=False)


## 10. Inspect the final records

Review a small set of the fields that will be most useful when the source-system records are created.


In [ ]:
ground_truth[
    [
        "ground_truth_id",
        "npi",
        "first_name",
        "middle_name",
        "last_name",
        "credential",
        "practice_city",
        "practice_state",
        "primary_taxonomy_code"
    ]
].head(20)


In [ ]:
ground_truth.sample(10, random_state=RANDOM_SEED)


In [ ]:
ground_truth.info()


## 11. Save the ground-truth CSV

This file should stay unchanged while the EHR, HR, and credentialing datasets are generated from it. Those datasets can contain altered or incomplete values, while this table remains the reference used to evaluate matching later in the project.


In [ ]:
GROUND_TRUTH_FILE = (
    GROUND_TRUTH_DIR / "provider_ground_truth.csv"
)

ground_truth.to_csv(
    GROUND_TRUTH_FILE,
    index=False
)

print(f"Saved ground truth to: {GROUND_TRUTH_FILE}")
